In [8]:
import tensorflow as tf
#import tensorflow_decision_forests as tfdf
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt

from sklearn.svm import SVC 
from sklearn.svm import SVC
from sklearn.naive_bayes import GaussianNB
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier, AdaBoostClassifier, StackingClassifier
#import xgboost as xgb
#from catboost import CatBoostClassifier
#from lightgbm import LGBMClassifier

from sklearn.impute import SimpleImputer
from sklearn.model_selection import train_test_split, GridSearchCV, RandomizedSearchCV
from sklearn.metrics import accuracy_score, f1_score, recall_score, precision_score, confusion_matrix
from sklearn.preprocessing import LabelEncoder, OneHotEncoder, StandardScaler, MinMaxScaler


train_df = pd.read_csv("train.csv")
train_df_copy = train_df.copy()
test_df = pd.read_csv("test.csv")
test_df_copy = test_df.copy()
print("Full train dataset shape is {}".format(train_df.shape))

Full train dataset shape is (8693, 14)


# Toutes les modifications de nos données

## Ajout de nouvelles variables

In [9]:
def age_group(df):
    age_group  = []
    for i in df["Age"]:
        if i<=4:
            age_group.append("Age_0-4")
        elif (i>4 and i<=12):
            age_group.append("Age_05-12")
        elif (i>12 and i<=18):
            age_group.append("Age_13-18")
        elif (i>18 and i<=25):
            age_group.append("Age_19-25")
        elif (i>25 and i<=32):
            age_group.append("Age_26-32")
        elif (i>32 and i<=50):
            age_group.append("Age_33_50")
        elif (i>50):
            age_group.append("Age_50+")
        else:
            age_group.append(np.nan)
        
    df["Age Group"] = age_group

age_group(train_df)
age_group(test_df)


def passagerid_new_features(df):
    df["Group"] = df["PassengerId"].apply(lambda x: int(x.split("_")[0]))
    df["Member"] = df["PassengerId"].apply(lambda x: int(x.split("_")[1]))

    x = df.groupby("Group")["Member"].count()
    y = set(x[x>1].index)

    df["Travelling_Solo"] = df["Group"].apply(lambda x : x not in y)
    df["Group_size"] = 0

    for i in x.items():
        df.loc[df["Group"]==i[0], "Group_size"] = i[1]
    df["Group_size"] = df["Group_size"].astype(int)

passagerid_new_features(train_df)
passagerid_new_features(test_df)

def cabin_new_feature(df):
    df["Cabin"].fillna("np.nan/np.nan/np.nan", inplace=True)
    
    df["Cabin_Deck"] = df["Cabin"].apply(lambda x: x.split("/")[0])
    df["Cabin_Number"] = df["Cabin"].apply(lambda x: x.split("/")[1])
    df["Cabin_Side"] = df["Cabin"].apply(lambda x: x.split("/")[2])
    
    # Remplacer les valeurs de chaîne 'np.nan' par des valeurs NaN de numpy
    cols = ["Cabin_Deck", "Cabin_Number", "Cabin_Side"]
    df[cols] = df[cols].replace("np.nan", np.nan)
    
    # Remplir les valeurs manquantes dans les nouvelles caractéristiques créées
    df["Cabin_Deck"].fillna(df["Cabin_Deck"].mode()[0], inplace=True)
    df["Cabin_Side"].fillna(df["Cabin_Side"].mode()[0], inplace=True)
    df["Cabin_Number"] = pd.to_numeric(df["Cabin_Number"], errors='coerce')  # Conversion en numérique
    df["Cabin_Number"].fillna(df["Cabin_Number"].median(), inplace=True)

cabin_new_feature(train_df)
cabin_new_feature(test_df)

def cabin_regions(df):
    df["Cabin_Region1"] = (df["Cabin_Number"]<300)
    df["Cabin_Region2"] = (df["Cabin_Number"]>=300) & (df["Cabin_Number"]<600)
    df["Cabin_Region3"] = (df["Cabin_Number"]>=600) & (df["Cabin_Number"]<900)
    df["Cabin_Region4"] = (df["Cabin_Number"]>=900) & (df["Cabin_Number"]<1200)
    df["Cabin_Region5"] = (df["Cabin_Number"]>=1200) & (df["Cabin_Number"]<1500)
    df["Cabin_Region6"] = (df["Cabin_Number"]>=1500)

cabin_regions(train_df)
cabin_regions(test_df)

exp_cols = ["RoomService","FoodCourt","ShoppingMall","Spa","VRDeck"]
def new_exp_features(df):
    df["Total Expenditure"] = df[exp_cols].sum(axis=1)
    df["No Spending"] = (df["Total Expenditure"]==0)

new_exp_features(train_df)
new_exp_features(test_df)

def expenditure_category(df):
    expense_category = []   
    for i in df["Total Expenditure"]:
        if i==0:
            expense_category.append("No Expense")
        elif (i>0 and i<=716):
            expense_category.append("Low Expense")
        elif (i>716 and i<=1441):
            expense_category.append("Medium Expense")
        elif (i>1441):
            expense_category.append("High Expense")
    df["Expenditure Category"] = expense_category

expenditure_category(train_df)
expenditure_category(test_df)

def is_in_a_group(row):
    if pd.isnull(row['Group']):
        return 0
    else:
        return 1
train_df['InGroup'] = train_df.apply(is_in_a_group, axis=1)
test_df['InGroup'] = test_df.apply(is_in_a_group, axis=1)

def fill_destination(df):
    # Remplir les valeurs manquantes dans chaque groupe avec la destination la plus fréquente dans ce groupe
    df['Destination'] = df.groupby('Group')['Destination'].transform(lambda x: x.fillna(x.mode()[0] if not x.mode().empty else x))

    # Remplir les valeurs manquantes restantes avec la destination la plus fréquente dans l'ensemble du DataFrame
    df['Destination'] = df['Destination'].fillna(df['Destination'].mode()[0])
fill_destination(train_df)
fill_destination(test_df)


## Remplissage des données manquantes

Cette partie est encore très naïve. IL faudra sûrement s'y attarder davantage dans le futur pour améliorer la précision de notre modèle.

In [10]:
cat_cols = train_df.select_dtypes(include=["object","bool"]).columns.tolist()
cat_cols.remove("Transported")
num_cols = train_df.select_dtypes(include=["int","float"]).columns.tolist()

def fill_missingno(df):
    df[cat_cols] = SimpleImputer(strategy="most_frequent").fit_transform(df[cat_cols])
    df[num_cols] = SimpleImputer(strategy="median").fit_transform(df[num_cols])

fill_missingno(train_df)
fill_missingno(test_df)

## Suppression des variables maintenant inutiles

In [11]:
pass_df = test_df[["PassengerId"]]
cols = ["PassengerId","Cabin","Name","Cabin_Number"]

train_df.drop(columns =cols, inplace=True)
test_df.drop(columns=cols, inplace=True)


## Traitement final : Encodage One-Hot et Label Encoding

In [12]:
nominal_cat_cols_one_Hot = ["HomePlanet","Destination"]

train_df = pd.get_dummies(train_df, columns= nominal_cat_cols_one_Hot)
test_df = pd.get_dummies(test_df, columns = nominal_cat_cols_one_Hot)

ordinal_cat_cols_Label = ["CryoSleep","VIP","Travelling_Solo","Cabin_Deck","Cabin_Side","Cabin_Region1","Cabin_Region2",
                    "Cabin_Region3","Cabin_Region4","Cabin_Region5","Cabin_Region6","Age Group","No Spending",
                    "Expenditure Category"]

binary_cols = [col for col in train_df.columns if train_df[col].dropna().unique().size == 2]
new_binary_cols = [col for col in binary_cols if col not in ordinal_cat_cols_Label]

ordinal_cat_cols_Label.extend(new_binary_cols)
ordinal_cat_cols_Label.remove("Transported")

train_df[ordinal_cat_cols_Label] = train_df[ordinal_cat_cols_Label].apply(LabelEncoder().fit_transform)
test_df[ordinal_cat_cols_Label] = test_df[ordinal_cat_cols_Label].apply(LabelEncoder().fit_transform)

## Pour commencer à travailler

Il est temps d'initialiser X et Y, et on fait à présent un partage des données entre X_train et X_test.
On fait également une normalisation des données : on aura donc le choix entre X_train et X_train_scaled pour construire notre modèle de prédiction.

In [13]:
Y = train_df["Transported"]
X = train_df.drop(columns=["Transported"])

X_scaled = StandardScaler().fit_transform(X)
test_df_scaled = StandardScaler().fit_transform(test_df)

X_train, X_test, Y_train, Y_test = train_test_split(X,Y,test_size=0.2,random_state=0)

X_train_scaled, X_test_scaled, Y_train_scaled, Y_test_scaled = train_test_split(X_scaled,Y,test_size=0.2,random_state=0)

## Suppression des variables créées inutiles

In [14]:
del binary_cols, cat_cols, cols, exp_cols, new_binary_cols, nominal_cat_cols_one_Hot, num_cols, ordinal_cat_cols_Label, Y_train_scaled, Y_test_scaled

# Les modèles

C'est là qu'on peut enfin tester nos modèles

In [15]:
from sklearn.linear_model import LogisticRegression

# Create an instance of Logistic Regression
regression_model = LogisticRegression(random_state=0)

# Fit the regression model using X_train_scaled and Y_train
regression_model.fit(X_train_scaled, Y_train)

# Predict the values of X_test_scaled
Y_pred = regression_model.predict(X_test_scaled)

# Print the accuracy of the model
print("Accuracy: ", accuracy_score(Y_test, Y_pred))

Accuracy:  0.7889591719378953


<h2>Optimisation des hyperparamètres</h2>

In [16]:
Y_test_pred = regression_model.predict(test_df_scaled)

#Créer le data frame de résultats
resultat_df = pd.DataFrame({
    'PassengerId':test_df_copy['PassengerId'],
    'Transported':Y_test_pred
})

resultat_df.to_csv('resultat.csv', index=False)

ça prend environ 1min40 pour tourner
(le bout en dessous)

In [17]:
from sklearn.model_selection import RandomizedSearchCV

model = RandomForestClassifier()

param_grid = {
    'n_estimators': [100, 200, 300, 400, 500],
    'max_depth': [10, 20, 30, 40, 50, 60, 70, 80, 90, 100, None],
    'max_features': ['auto', 'sqrt'],
    'min_samples_split': [2, 5, 10],
    'min_samples_leaf': [1, 2, 4],
    'bootstrap': [True, False]
}

random_search = RandomizedSearchCV(estimator=model, param_distributions=param_grid, n_iter=100, cv=3, verbose=2, random_state=42, n_jobs=-1)

random_search.fit(X_train_scaled, Y_train)


Fitting 3 folds for each of 100 candidates, totalling 300 fits


KeyboardInterrupt: 

In [ ]:
Y_test_pred_2 = random_search.predict(test_df_scaled)

#Créer le data frame de résultats
resultat_df2 = pd.DataFrame({
    'PassengerId':test_df_copy['PassengerId'],
    'Transported':Y_test_pred_2
})

resultat_df2.to_csv('resultat_2.csv', index=False)

<h2>Utilisaion de SVM</h2>

<p>(j'ai arreter le prgromme car il prenait trop de temps à tourner, plus de 5 heures)</p>
-> réduction du nombre de valeurs pour kernel, C et gamma

Pb: avec kernel = poly on obtient des floats et donc la précision est de 0.0

In [ ]:
from sklearn.svm import SVR
from sklearn.model_selection import train_test_split, GridSearchCV
import pandas as pd

param_grid_svm ={
    'kernel':['poly'],
    'C': [0.1, 1, 10],
    'gamma': ['scale', 'auto'],
    'epsilon':[0.1,0.3,0.5]
}

# Initialiser le modèle
model_svm = SVR()

grid_search_svm = GridSearchCV(model_svm,param_grid_svm,cv=5,scoring='neg_mean_squared_error')
# Entraîner le modèle
grid_search_svm.fit(X_train_scaled, Y_train)

Y_test_pred = grid_search_svm.predict(test_df_scaled)

#Créer le data frame de résultats
resultat_df3 = pd.DataFrame({
    'PassengerId':test_df_copy['PassengerId'],
    'Transported':Y_test_pred
})

resultat_df3.to_csv('resultat_3.csv', index=False)

SVM en rendomsearchCV pour l'opti des hyperparamètres

j'ai arreté le code au bout de 330 minutes

In [21]:
from sklearn.svm import SVC
from sklearn.model_selection import RandomizedSearchCV
import pandas as pd

# Initialiser le modèle
model_svm = SVC()

# Définir les paramètres à optimiser
param_grid = {
    'gamma':[0.1,1,10,100],
    'C':[2,3,4,5,6],
    'kernel': ['linear', 'poly', 'rbf', 'sigmoid']
}

# Initialiser la recherche aléatoire
random_search = RandomizedSearchCV(model_svm, param_grid, cv=5, n_iter=300)

# Entraîner le modèle
random_search.fit(X_train_scaled, Y_train)

# Afficher les meilleurs paramètres
print("Meilleurs paramètres : ", random_search.best_params_)

# Faire une prédiction avec le meilleur modèle
Y_test_pred = random_search.predict(test_df_scaled)

# Créer le data frame de résultats
resultat_df = pd.DataFrame({
    'PassengerId': test_df_copy['PassengerId'],
    'Transported': Y_test_pred
})

resultat_df.to_csv('resultat_svm.csv', index=False)

c:\Users\theot\AppData\Local\Programs\Python\Python311\Lib\site-packages\sklearn\model_selection\_search.py:318: UserWarning: The total space of parameters 80 is smaller than n_iter=300. Running 80 iterations. For exhaustive searches, use GridSearchCV.
  warnings.warn(


<h4>Nouvelle tentaive d'optimisation des hyperparametres du SVM

<h3>Nearest Neighbors</h3>

precision = 0.74982 (avant opti des hyperparamètres)

In [ ]:
from sklearn.neighbors import KNeighborsClassifier
from sklearn.model_selection import train_test_split
import pandas as pd

# Initialiser le modèle avec le nombre de voisins que vous voulez - par exemple, 3
model_knn = KNeighborsClassifier(n_neighbors=3)

# Entraîner le modèle
model_knn.fit(X_train_scaled, Y_train)

# Faire une prédiction
Y_test_pred = model_knn.predict(test_df_scaled)

# Créer le data frame de résultats
resultat_df = pd.DataFrame({
    'PassengerId': test_df_copy['PassengerId'],
    'Transported': Y_test_pred
})

resultat_df.to_csv('resultat_knn.csv', index=False)

(opti des hyperparamètres) precision = 0.77367

In [ ]:
from sklearn.neighbors import KNeighborsClassifier
from sklearn.model_selection import GridSearchCV
import pandas as pd

# Initialiser le modèle
model_knn = KNeighborsClassifier()

# Définir les paramètres à optimiser
param_grid = {
    'n_neighbors': list(range(1, 15)),
    'weights': ['uniform', 'distance'],
    'metric': ['euclidean', 'manhattan', 'minkowski']
}

# Initialiser la recherche sur grille
grid_search = GridSearchCV(model_knn, param_grid, cv=5)

# Entraîner le modèle
grid_search.fit(X_train_scaled, Y_train)

# Afficher les meilleurs paramètres
print("Meilleurs paramètres : ", grid_search.best_params_)

# Faire une prédiction avec le meilleur modèle
Y_test_pred = grid_search.predict(test_df_scaled)

# Créer le data frame de résultats
resultat_df = pd.DataFrame({
    'PassengerId': test_df_copy['PassengerId'],
    'Transported': Y_test_pred
})

resultat_df.to_csv('resultat_knn.csv', index=False)

KeyboardInterrupt: 

<h2>Logistic Regression</h2>

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import GridSearchCV
import pandas as pd

# Initialiser le modèle
model_lr = LogisticRegression()

# Définir les paramètres à optimiser
param_grid = {
    'C': [0.01,1, 10, 100],
    'penalty': ['l1', 'l2', 'none','elasticnet'],
    'max_iter': list(range(100,800,100)),
    'solver': ['newton-cg', 'lbfgs', 'liblinear', 'sag', 'saga'],
    'l1_ratio': [0.1, 0.5, 0.9]
}

# Initialiser la recherche sur grille
grid_search = GridSearchCV(model_lr, param_grid, cv=5)

# Entraîner le modèle
grid_search.fit(X_train_scaled, Y_train)

# Afficher les meilleurs paramètres
print("Meilleurs paramètres : ", grid_search.best_params_)

# Faire une prédiction avec le meilleur modèle
Y_test_pred = grid_search.predict(test_df_scaled)

# Créer le data frame de résultats
resultat_df = pd.DataFrame({
    'PassengerId': test_df_copy['PassengerId'],
    'Transported': Y_test_pred
})

resultat_df.to_csv('resultat_lr.csv', index=False)

c:\Users\theot\AppData\Local\Programs\Python\Python311\Lib\site-packages\sklearn\linear_model\_logistic.py:1175: UserWarning: l1_ratio parameter is only used when penalty is 'elasticnet'. Got (penalty=l1)
  warnings.warn(
c:\Users\theot\AppData\Local\Programs\Python\Python311\Lib\site-packages\sklearn\linear_model\_logistic.py:1175: UserWarning: l1_ratio parameter is only used when penalty is 'elasticnet'. Got (penalty=l1)
  warnings.warn(
c:\Users\theot\AppData\Local\Programs\Python\Python311\Lib\site-packages\sklearn\linear_model\_logistic.py:1175: UserWarning: l1_ratio parameter is only used when penalty is 'elasticnet'. Got (penalty=l1)
  warnings.warn(
c:\Users\theot\AppData\Local\Programs\Python\Python311\Lib\site-packages\sklearn\linear_model\_logistic.py:1175: UserWarning: l1_ratio parameter is only used when penalty is 'elasticnet'. Got (penalty=l1)
  warnings.warn(
c:\Users\theot\AppData\Local\Programs\Python\Python311\Lib\site-packages\sklearn\linear_model\_logistic.py:1175:

Meilleurs paramètres :  {'C': 1, 'l1_ratio': 0.1, 'max_iter': 200, 'penalty': 'l1', 'solver': 'saga'}


c:\Users\theot\AppData\Local\Programs\Python\Python311\Lib\site-packages\sklearn\linear_model\_sag.py:350: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(


<h2>Ensemble learning</h2>

In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import GridSearchCV
import pandas as pd

# Initialiser le modèle
model_rf = RandomForestClassifier()

# Définir les paramètres à optimiser
param_grid = {
    'n_estimators': [100, 200, 500],
    'max_features': ['auto', 'sqrt', 'log2'],
    'max_depth' : [4,5,6,7,8],
    'criterion' :['gini', 'entropy']
}

# Initialiser la recherche sur grille
grid_search = GridSearchCV(model_rf, param_grid, cv=5)

# Entraîner le modèle
print("Meilleurs paramètres : ", grid_search.best_params_)


# Faire une prédiction avec le meilleur modèle
Y_test_pred = grid_search.predict(test_df_scaled)

# Créer le data frame de résultats
resultat_df = pd.DataFrame({
    'PassengerId': test_df_copy['PassengerId'],
    'Transported': Y_test_pred
})

resultat_df.to_csv('resultat_rf.csv', index=False)

c:\Users\theot\AppData\Local\Programs\Python\Python311\Lib\site-packages\sklearn\model_selection\_validation.py:547: FitFailedWarning: 
150 fits failed out of a total of 450.
The score on these train-test partitions for these parameters will be set to nan.
If these failures are not expected, you can try to debug them by setting error_score='raise'.

Below are more details about the failures:
--------------------------------------------------------------------------------
150 fits failed with the following error:
Traceback (most recent call last):
  File "c:\Users\theot\AppData\Local\Programs\Python\Python311\Lib\site-packages\sklearn\model_selection\_validation.py", line 895, in _fit_and_score
    estimator.fit(X_train, y_train, **fit_params)
  File "c:\Users\theot\AppData\Local\Programs\Python\Python311\Lib\site-packages\sklearn\base.py", line 1467, in wrapper
    estimator._validate_params()
  File "c:\Users\theot\AppData\Local\Programs\Python\Python311\Lib\site-packages\sklearn\base

Meilleurs paramètres :  {'criterion': 'gini', 'max_depth': 8, 'max_features': 'sqrt', 'n_estimators': 200}


Optimisations des hyperparamètres avec une random search et en utilisant les info sur les meilleurs paramètres obtenue lors de la grid search

In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import RandomizedSearchCV
import pandas as pd

# Initialiser le modèle
model_rf = RandomForestClassifier()

# Définir les paramètres à optimiser
param_grid = {
    'n_estimators': [150, 175, 200,225,250],
    'max_features': ['auto', 'sqrt', 'log2'],
    'max_depth' : list(range(4, 12)),
    'min_samples_split': [2, 5, 10],
    'min_samples_leaf': [1, 2, 4],
    'bootstrap': [True, False],
    'criterion' :['gini', 'entropy']
}

# Initialiser la recherche aléatoire
random_search = RandomizedSearchCV(model_rf, param_grid, cv=5, n_iter=100)

# Entraîner le modèle
random_search.fit(X_train_scaled, Y_train)

# Afficher les meilleurs paramètres
print("Meilleurs paramètres : ", random_search.best_params_)

# Faire une prédiction avec le meilleur modèle
Y_test_pred = random_search.predict(test_df_scaled)

# Créer le data frame de résultats
resultat_df = pd.DataFrame({
    'PassengerId': test_df_copy['PassengerId'],
    'Transported': Y_test_pred
})

resultat_df.to_csv('resultat_rf_rd.csv', index=False)

c:\Users\theot\AppData\Local\Programs\Python\Python311\Lib\site-packages\sklearn\model_selection\_validation.py:547: FitFailedWarning: 
140 fits failed out of a total of 500.
The score on these train-test partitions for these parameters will be set to nan.
If these failures are not expected, you can try to debug them by setting error_score='raise'.

Below are more details about the failures:
--------------------------------------------------------------------------------
140 fits failed with the following error:
Traceback (most recent call last):
  File "c:\Users\theot\AppData\Local\Programs\Python\Python311\Lib\site-packages\sklearn\model_selection\_validation.py", line 895, in _fit_and_score
    estimator.fit(X_train, y_train, **fit_params)
  File "c:\Users\theot\AppData\Local\Programs\Python\Python311\Lib\site-packages\sklearn\base.py", line 1467, in wrapper
    estimator._validate_params()
  File "c:\Users\theot\AppData\Local\Programs\Python\Python311\Lib\site-packages\sklearn\base

Meilleurs paramètres :  {'n_estimators': 225, 'min_samples_split': 2, 'min_samples_leaf': 2, 'max_features': 'log2', 'max_depth': 11, 'criterion': 'gini', 'bootstrap': False}


<h2>Bayesian</h2>

In [ ]:
from sklearn.naive_bayes import GaussianNB
from sklearn.model_selection import GridSearchCV
import numpy as np
import pandas as pd

# Initialiser le modèle
model_nb = GaussianNB()

# Définir les paramètres à optimiser
param_grid = {
    'var_smoothing': np.logspace(0,-9, num=100)
}

# Initialiser la recherche sur grille
grid_search = GridSearchCV(model_nb, param_grid, cv=5)

# Entraîner le modèle
grid_search.fit(X_train_scaled, Y_train)

# Afficher les meilleurs paramètres
print("Meilleurs paramètres : ", grid_search.best_params_)

# Faire une prédiction avec le meilleur modèle
Y_test_pred = grid_search.predict(test_df_scaled)

# Créer le data frame de résultats
resultat_df = pd.DataFrame({
    'PassengerId': test_df_copy['PassengerId'],
    'Transported': Y_test_pred
})

resultat_df.to_csv('resultat_nb.csv', index=False)

Meilleurs paramètres :  {'var_smoothing': 1.0}
